In [ ]:
Alma Dollente
11/25/26


## Assignment Module 8 Lab 2: Predicting House Prices using Linear Regression with Statsmodels

### Background

Accurately predicting house prices is essential in the real estate market. In this assignment, you will use linear regression to predict house prices based on various features such as the number of rooms, population density, and median income.

### Dataset

You will use the California Housing dataset, which contains information about houses in different locations in California. The dataset includes 20,640 observations and 8 variables, including:

- `MedInc`: Median income in the block
- `HouseAge`: Median house age in the block
- `AveRooms`: Average number of rooms per dwelling
- `AveBedrms`: Average number of bedrooms per dwelling
- `Population`: Block population
- `AveOccup`: Average house occupancy
- `Latitude`: Latitude coordinate of the block
- `Longitude`: Longitude coordinate of the block

### Task

Your task is to predict house prices in California using linear regression with the Statsmodels library. You will need to perform the following steps:

1. Load the California Housing dataset into a Pandas DataFrame.
2. Split the dataset into training and testing sets, with a 70/30 split.
3. Fit a linear regression model using the Statsmodels library to the training data.
4. Predict house prices for the testing set.
5. Evaluate the performance of the model using the following error checking metrics:
   - R-squared (R2)
   - Mean Squared Error (MSE)
   - Mean Absolute Error (MAE)
   - Akaike Information Criterion (AIC)
6. Analyze the performance of the model and provide recommendations for improvement.

### Instructions

1. Load the California Housing dataset into a Pandas DataFrame.
2. Split the dataset into training and testing sets using a 70/30 split. Set the random seed to 42 for reproducibility.
3. Import the necessary libraries:
   - `import statsmodels.api as sm`
   - `from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error`
4. Create the target variable (`y`) and feature variables (`X`) from the DataFrame.
5. Add a constant column of 1s to the feature variables using `sm.add_constant()`.
6. Fit a linear regression model to the training data using `sm.OLS()`. Use the constant-augmented feature variables and the target variable as arguments.
7. Print the summary of the regression model using `model.summary()`. Note the AIC value.
8. Predict house prices for the testing set using `model.predict()` on the constant-augmented feature variables of the testing set.
9. Calculate and print the R-squared, MSE, and MAE using the predicted values and the actual values of the testing set.
10. Analyze the performance of the model and provide recommendations for improvement. Consider the R-squared, MSE, MAE, and AIC values.
11. Include detailed comments throughout your code to explain each step.

### Submission Instructions

Please submit a Jupyter notebook containing your code, output, and analysis. Make sure to include the following sections in your notebook:

1. Introduction
2. Dataset Description
3. Data Preprocessing
4. Model Training and Evaluation
5. Performance Analysis and Recommendations
6. Conclusion

Your notebook should be well-documented and easy to follow, with clear explanations of the steps you took and the results you obtained. Make sure to comment your code and include markdown cells explaining your thought process and conclusions.

# Predicting House Prices in California using Linear Regression with Statsmodels

## 1. Introduction
Estimating residential property values accurately is critical for real estate pricing, investment analysis, and mortgage underwriting. An evaluation of an Ordinary Least Squares (OLS) linear regression model using the `statsmodels` library enables this notebook to predict median house values across California using socioeconomic, structural, and geographic census block group attributes.  

The six analysis stages: 
1. Describing the California Housing dataset
2. Preprocessing and splitting the data into 70% training and 30% testing sets.
3. Fit an intercept-augmented `statsmodels` OLS regression model
4. Generating predictions from the test dataset
5. Evaluating performance across $R^2$, Mean Squared Error (MSE), Mean Absolute Error (MAE), and Akaike Information Criterion (AIC)
6. Diagnosing model limitations for improvements

In [1]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# load the California Housing dataset into a Pandas DataFrame
ca_housing = fetch_california_housing(as_frame=True)
df = ca_housing.frame

## 2. Dataset Description

In [2]:
print(ca_housing.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib:
https://lib.stat.cmu.edu/datasets/houses.zip

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per census
block group. A block g

## 3. Data Preprocessing

The DataFrame is inspected to verify its dimensions, confirm that all features are stored as numeric data types (df.dtypes), and check for missing values or duplicate records.

In [3]:
# inspect the dataset
df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


All 20,640 records across 9 columns are stored as numeric datatypes, with no missing values and zero duplicate rows. Consequently, no imputation or categorical encoding is required, and features are left unscaled so the baseline OLS coefficients remain directly interpretable in their original units.

Next, eight predictor features (`X`) are separated from the target variable (`y = MedHouseVal`) and the data set is split into 70% training subset and 30% testing subset using `random_state=42` for reproducibility. An explicit constant column of `1's` is prepended to both predictor matrices using `sm.add_constant()` because `statsmodels` OLS regression does not automatically include an intercept. 

In [4]:
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# drop the target feature from the training portion, and assign it to the target variable
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

# split the data set, 70% for training, 30% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# add a constant column of 1s to both training and testing predictors for the OLS intercept
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# verify the shapes of each subset
X_train.shape, X_test.shape

((14448, 8), (6192, 8))

## 4. Model Training and Evaluation

`X_train_const`, the constant-augmented training feature matrix, and `y_train`, target vector, are fit into an OLS regression model to estimate the baseline linear relationship between the eight housing features and median house value (`MedHouseVal`). After inspecting the full `statsmodels` diagnostic summary and in-sample Akaike Information Criterion (AIC), predictions are made from `X_test_const` and assessed against `y_test` using $R^2$, Mean Squared Error (MSE), and Mean Absolute Error (MAE).

In [5]:
import pandas as pd

# fit the OLS linear regression model to the constant-augmented training data
model = sm.OLS(y_train, X_train_const).fit()

# print the full regression summary table (including coefficients, p-values, and AIC)
print(model.summary())

# make house price predictions on the constant-augmented testing set
y_pred = model.predict(X_test_const)

# calculate evaluation metrics on the testing set
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
aic = model.aic

# return test set performance metrics and training AIC
pd.Series({
    "R-squared (R2)": round(r2_score(y_test, y_pred), 4),
    "Mean Squared Error (MSE)": round(mean_squared_error(y_test, y_pred), 4),
    "Mean Absolute Error (MAE)": round(mean_absolute_error(y_test, y_pred), 4),
    "Akaike Information Criterion (AIC)": round(model.aic, 2),
}).to_frame(name="Value")

                            OLS Regression Results                            
Dep. Variable:            MedHouseVal   R-squared:                       0.609
Model:                            OLS   Adj. R-squared:                  0.609
Method:                 Least Squares   F-statistic:                     2815.
Date:                Wed, 25 Nov 2026   Prob (F-statistic):               0.00
Time:                        21:23:06   Log-Likelihood:                -15823.
No. Observations:               14448   AIC:                         3.166e+04
Df Residuals:                   14439   BIC:                         3.173e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -37.0562      0.779    -47.577      0.0

,Value
R-squared (R2),0.5958
Mean Squared Error (MSE),0.5306
Mean Absolute Error (MAE),0.5272
Akaike Information Criterion (AIC),31664.7100


## 5. Performance Analysis and Recommendations

### Performance Analysis

The `statsmodels` OLS model accounts for 60.9% of training variance in `MedHouseVal` ($R^2 = 0.609$, $\text{Adjusted } R^2 = 0.609$, $F = 2815, p < 0.001$) and achieves a test set $R^2$ of 59.58%, revealing consistent generalization without signs of overfitting. Leaving 40.42% of test price variance unexplained—alongside a test MAE of `0.5272` (`$52,720`), an MSE of `0.5306` ($\text{RMSE} \approx 0.7284$, or `~$72,840`), and a high training set AIC of `31,664.71`, indicating substantial errors on high-value properties.

Each OLS coefficient ($\beta$) estimates the expected change in `MedHouseVal` per one-unit predictor increase, holding other features constant, revealing key drivers and structural limits:

* **Economic and Spatial Drivers:** `MedInc` ($\beta = 0.4458, t = 89.283, p < 0.001$) is the strongest positive predictor, adding ~$44,580 per $10,000 in income. Negative slopes for `Latitude` ($\beta = -0.4185$) and `Longitude` ($\beta = -0.4337$) capture California's coastal gradient—prices rise moving south and west toward the Pacific—but a linear plane cannot separate high-priced northern hubs like San Francisco from lower-priced inland Central Valley cities at similar latitudes. Meanwhile, `Population` ($\beta = -7.76 \times 10^{-7}, p = 0.890$) adds no significant linear signal.
  
* **Multicollinearity and Scale Distortion:** A large condition number ($2.37 \times 10^5$) implies coefficient estimates are highly sensitive to small input fluctuations due to unscaled features and overlapping predictors. Wide scale disparities such as high Population and single-digit room ratios in combination with strong collinearity between `AveRooms` ($\beta = -0.1221$) and `AveBedrms` ($\beta = +0.7786$), where total room count has to take a counterintuitive negative slope to keep the number of bedrooms constant.
  
* **Non-Normal Residuals and Target Capping:** Positive skew ($1.081$), high kurtosis ($6.091$), and a Jarque-Bera statistic of $8566.50$ ($p < 0.001$) demonstrate that errors do not follow a normal distribution, largely because of a $500,001 upper-bound truncation, where linear projections systematically underpredict luxury coastal markets.

### Recommendations for Improvement

1. **Engineer Ratios and Spatial Features:** Construct new predictors like giving AveRooms and AveBedrms their own ratio (`AveBedrms / AveRooms`), remove insignificant predictors like Population, and add distance-to-coast or distance-to-metro-hub features for spatial effects that aren't captured from coordinates alone.
2. **Standardize Scales and Regularize:** Fit `StandardScaler` on `X_train` within a `scikit-learn` pipeline to eliminate scale instability, and to stabilize collinear feature weights, test with **Ridge** or **ElasticNet** regression.
3. **Adopt Non-Linear Ensemble Models:** Use tree-based ensembles like **Random Forest**, **Gradient Boosting**, or **XGBoost** that are capable of capturing non-linear interactions between income, location, and housing density without assuming a strictly linear plane.
4. **Address Right-Censoring and Target Skew:** Test a log-transformed target ($\log(\text{MedHouseVal})$) or run a sensitivity check  without the $500,001 cap, preventing upper-bound ceiling distortion from biasing slope estimates.

## 6. Conclusion

An Ordinary Least Squares (OLS) baseline model was trained and evaluated with `statsmodels` on the California Housing dataset as an interpretable regional benchmark for predicting median home values. Across 6,192 unseen test cases, the model achieved an $R^2$ of `0.5958`, an MAE of `0.5272` (`$52,720`), an MSE of `0.5306`, and an in-sample AIC of `31,664.71`.

Real estate pricing violates several core linear assumptions so the phrasing stays direct and natural., even while the model shows several intriguing regional and economic trends (such as the coastal price gradient in California and the overwhelming role of household income). The power of such a basic linear model is weakened by the price cap, housing ratio overlaps, unscaled features, and heavy-tailed errors.

In conclusion, OLS offers a useful starting point for determining the primary factors influencing price, but careful feature set selection, standard scaling, and non-linear regression approaches are necessary for high-quality modeling of home prices.